# Google Colab (Skip locally)

In [ ]:
#@title Mount your Google Drive
# If you run this notebook locally or on a cluster (i.e. not on Google Colab)
# you can delete this cell which is specific to Google Colab. You may also
# change the paths for data/logs in Arguments below.
%matplotlib inline
%load_ext autoreload
%autoreload 2

from google.colab import drive
drive.mount('/content/gdrive')


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Mounted at /content/gdrive


## Link source code

In [ ]:
#@title Link via google drive
#@markdown Enter the path to the assignment folder in your Google Drive
# If you run this notebook locally or on a cluster (i.e. not on Google Colab)
# you can delete this cell which is specific to Google Colab. You may also
# change the paths for data/logs in Arguments below.
import sys
import os
import shutil

folder = "/content/gdrive/MyDrive/IFT6135/HW2_2025/src" #@param {type:"string"}
!ln -Ts "$folder" /content/src 2> /dev/null

# Add the assignment folder to Python path
if '/content/src' not in sys.path:
  sys.path.insert(0, '/content/src')

In [ ]:
#@title Link via zip upload
from google.colab import files
import sys
import os
import shutil
import warnings

# 1️⃣ Upload your ZIP containing only your Python code (it may contain a top-level 'src' folder)
uploaded = files.upload()  # choose your zip
if not uploaded:
    raise FileNotFoundError("No ZIP file uploaded.")
zip_name = next(iter(uploaded.keys()))
print(f"✔ Uploaded: {zip_name}")

# 2️⃣ Prepare /content/src
base_dir = "/content/src"
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
os.makedirs(base_dir, exist_ok=True)

# 3️⃣ Extract into /content/src
!unzip -q "{zip_name}" -d "{base_dir}"
print("✔ Extracted into /content/src/")

# 4️⃣ If there's a single top-level folder inside (e.g. /content/src/src/), flatten it
entries = os.listdir(base_dir)
if len(entries) == 1 and os.path.isdir(os.path.join(base_dir, entries[0])):
    nested = os.path.join(base_dir, entries[0])
    for fname in os.listdir(nested):
        shutil.move(os.path.join(nested, fname), base_dir)
    shutil.rmtree(nested)
    print(f"✔ Flattened nested folder `{entries[0]}` into /content/src/")

# 5️⃣ Add /content/src to Python path for top-level imports
if base_dir not in sys.path:
    sys.path.insert(0, base_dir)
    print("✔ Added '/content/src' to sys.path")


Saving src.zip to src.zip
✔ Uploaded: src.zip
✔ Extracted into /content/src/
✔ Flattened nested folder `src` into /content/src/
✔ Added '/content/src' to sys.path


## Setting up virtual environment

In [ ]:
#@title First time setup
!pip install virtualenv
!virtualenv /content/gdrive/MyDrive/qaml_env
!source /content/drive/MyDrive/qaml_env/bin/activate; pip install -r requirements.txt

In [ ]:
#@title Connect to virtual environment
import sys
# add the path of the virtual environment site-packages to colab system path
sys.path.append("/content/drive/MyDrive/qaml_env/lib/python3.8/site-packages")

## Checks

In [ ]:
import warnings
try:
    import torch
    if not torch.cuda.is_available():
        warnings.warn('CUDA is not available.')
    else:
        print(f'Using device: {torch.cuda.get_device_name(0)}')
except ImportError:
    pass

# Imports & Setup

In [1]:
%matplotlib inline
import warnings

import os
import random
import argparse
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from collections import defaultdict
import pandas as pd

from visualisation.AttacksVisualisation import (
    plot_attack_effect_on_accuracy,
    plot_transfer_attack_effects,
)
from visualisation.TrainingVisualisation import (
    plot_models_together,
    plot_training_curves,
    generate_gradient_flow_plots,
)
from arguments import Arguments
from models.classical import (
    ResNet18,
    MLPMixer,
    MLP,
    ConvNet,
    LeNet,
)
from models.hybrid import (
    HQuanv,
    HQNN,
    HQE,
    DimVQC,
    AmpVQC,
)

import json
import time

from main import train_model, run_attacks, main

from utils import seed_experiment, calculate_accuracy
from data.DatasetManagerFactory import DatasetManagerFactory
from attacks import PGDAttack, FGSMAttack
from attacks.TransferAttack import transfer_attack
from metrics.LoggingService import LoggingService

# Experiments

## Experiment #1

In [ ]:
model_name = "HQNN"  #@param {type:"string"}
model_cls = AmpVQC  #@param {type:"string"}
#path = f"C:\\Users\\fwilh\\Desktop\\QMLBenchmark\\logs\\test_models\\2025-06-27@14-35-01\\{model_name}\\{model_name}_1\\0"
path = f"C:\\Users\\fwilh\\Desktop\\QMLBenchmark\\logs\\test_models\\2025-06-27@19-00-06\\{model_name}\\{model_name}_1\\0\\final.ckpt"
path = "C:\\Users\\fwilh\\Desktop\\QMLBenchmark\\logs\\test_models\\2025-06-30@14-43-55\\AmpVQC\\AmpVQC_1\\0\\final.ckpt"

# load the model
model = model_cls.load_from_checkpoint(path, map_location='cpu')

In [ ]:
#plot_training_result(cont, 42, 42)
#generate_gradient_flow_plots(cont)

## Experiment #2

In [ ]:
# Define a arguments and set global defaults
LoggingService("test_models")
seeds = [0, 42]
args = Arguments()
args.verbose = True
args.n_steps = 10
# args.num_workers = 0
# args.device = 'cuda'
# args.accelerator = 'cuda'
# args.devices = 1

args.eval_ratio = 0.4
args.train_ratio = 0.6

args.train_batch_size = 2**8
args.eval_batch_size = 2**11

args.train_log_interval = 0.5
args.eval_log_interval = 0.5

# 2) Define your list of model‐configurations: (name, class, kwargs)
model_specs = [
    ("ResNet18_1", ResNet18, {
        "input_size": (1, 28),
        "num_classes": 10,
    }),
    ("MLPMixer_1", MLPMixer, {
        "num_classes": 10,
        "input_size": (1, 28),
        "patch_size": 4,
        "embed_dim": 256,
        "num_blocks": 4,
        "drop_rate": 0.0,
        "activation": "gelu",
        "mlp_ratio": (0.5, 4.0),
    }),
    ("MLP_1", MLP, {
        "input_size": (1, 28),
        "hidden_sizes": [1024, 512, 64, 64],
        "num_classes": 10,
        "activation": "relu",
    }),
    ("ConvNet_1", ConvNet, {
        "input_size": (1, 28),
        "num_classes": 10,
    }),
    ("LeNet_1", LeNet, {
        "in_channels": 1,
        "classes": 10,
    }),
    ("HQNN_Parallel_1", HQNN, {
        "in_channels": 1,
        "classes": 10,
    }),
    ("HQuanv_1", HQuanv, {
        "in_channels": 1,
        "classes": 10,
        "img_size": 28,
        "n_qdepth": 2,
    }),
    ("HQE_1", HQE, {
        "input_size": 28 * 28,
        "hidden_sizes": [4096, 2048, 1024],
        "n_qdepth": 2,
        "classes": 10,
    }),
    ("DimVQC_1", DimVQC, {
        "input_size": (1, 28),
        "reduction_method": "pca",
        "reduced_dim": 1024,
        "n_qdepth": 2,
    }),
    ("AmpVQC_depth1", AmpVQC, {
        "input_size": 28 * 28,
        "n_qdepth": 2,
    }),
]

## Models Training

In [ ]:
trained_models = {}
for (model_name, model_cls, config) in model_specs:
    args.exp_name = model_name
    args.model = model_cls
    args.model_config = config

    trained_models[model_name] = train_model(args, seeds)
    plot_training_curves(model_name, trained_models[model_name])
    generate_gradient_flow_plots(list(trained_models[model_name].values())[0])

plot_models_together(trained_models)

>>> Starting experiment 'ResNet18_1' over seeds [0, 42]
(1/2); Seed = 0  ...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


ResNet18(
  (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (shortcut): Sequential()
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(6


  | Name    | Type              | Params | Mode 
------------------------------------------------------
0 | conv1   | Conv2d            | 576    | train
1 | bn1     | BatchNorm2d       | 128    | train
2 | layer1  | Sequential        | 147 K  | train
3 | layer2  | Sequential        | 525 K  | train
4 | layer3  | Sequential        | 2.1 M  | train
5 | layer4  | Sequential        | 8.4 M  | train
6 | linear  | Linear            | 5.1 K  | train
7 | relu    | ReLU              | 0      | train
8 | avgpool | AdaptiveAvgPool2d | 0      | train
9 | loss_fn | CrossEntropyLoss  | 0      | train
------------------------------------------------------
11.2 M    Trainable params
0         Non-trainable params
11.2 M    Total params
44.691    Total estimated model params size (MB)
72        Modules in train mode
0         Modules in eval mode


verbose             : True
n_steps             : 142
eval_ratio          : 0.4
train_ratio         : 0.6
train_batch_size    : 256
eval_batch_size     : 2048
train_log_interval  : 0.5
eval_log_interval   : 0.5
exp_name            : ResNet18_1
model               : <class 'models.classical.ResNet18.ResNet18'>
model_config        : {'input_size': (1, 28), 'num_classes': 10}
seed                : 0
 rootdir      : ../logs\test_models\2025-07-04@02-03-40\ResNet18\ResNet18_1\0
 train_size: 36000
 eval_size: 24000
accelerator: auto
devices: 1
num_workers: 0
num_threads: None
device: cpu


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\fwilh\.conda\envs\QAML\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\fwilh\.conda\envs\QAML\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

## Attacks

In [ ]:
# Run transfer attacks
# attacks_metrics = run_attacks(trained_models, args)
# plot_transfer_attack_effects(attacks_metrics,
#                             title_prefix="transfer")
# from pprint import pprint
# pprint(trained_models['HQNN_Parallel_1'][0].all_metrics, depth=2)
# metrics = trained_models['HQNN_Parallel_1'][0].all_metrics

## Final

In [ ]:
# prompt: code the function make archive who receives the dir to archive and the zipped file name

def make_archive(dir_to_archive, zipped_file_name):
    """
    Archives a directory into a zip file.

    Args:
        dir_to_archive (str): The path to the directory to be archived.
        zipped_file_name (str): The desired name for the output zip file (without extension).
    """
    try:
        shutil.make_archive(zipped_file_name, 'zip', dir_to_archive)
        print(f"Successfully created archive: {zipped_file_name}.zip")
    except Exception as e:
        print(f"Error creating archive: {e}")

# Example usage (you would call this function with the actual directory and file name)
# make_archive('/content/src', 'my_source_code')
# files.download('my_source_code.zip') # To download the created zip file

In [ ]:
make_archive('logs', 'test1')

Successfully created archive: test1.zip
